# Run an experiment on a Kaggle GPU

Accelerator: GPU T4 x1 or P100. Internet: on. Inputs: the private datasets `sixth-finger-bci-epochs` and, for fine-tuning, `sixth-finger-bci-pretrained`. To resume an interrupted grid, add the previous version's output as an input (Add Data → your notebook's output); Kaggle mounts it read-only under `/kaggle/input/<slug>/out/`, and cell 4 copies its `runs/` back.

The repository is the published one. It must be cloned with its tags: the runner refuses full-cohort runs unless the `prereg-v1` tag exists.

In [ ]:
EXPERIMENT = "configs/experiments/rq2_calib_s1.yaml"
METHODS = "eegnet_scratch,eegnet_pooled,eegnet_ea_ft,eegnet_ea_head"
SUBJECTS = "all"
REPO = "https://github.com/<owner>/sixth-finger-bci.git"

In [ ]:
import os
import subprocess

from kaggle_secrets import UserSecretsClient

token = UserSecretsClient().get_secret("GH_TOKEN")
url = REPO.replace("https://", f"https://{token}@")
# A full clone, not --depth 1: the pre-registration tag points at an older commit.
subprocess.run(["git", "clone", url, "repo"], check=True)
os.chdir("repo")
subprocess.run(["git", "fetch", "--tags", "origin"], check=True)
tags = subprocess.run(["git", "tag", "--list", "prereg-v1"], capture_output=True, text=True).stdout
assert "prereg-v1" in tags, "prereg-v1 tag missing: full-cohort runs would be refused"
print(subprocess.run(["git", "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout)

In [ ]:
# Kaggle already has a CUDA build of torch; install the rest of the pinned stack around it.
import sys

print(sys.version)
flag = "" if sys.version_info[:2] == (3, 12) else "--ignore-requires-python"
if flag:
    print("WARNING: the project is pinned to Python 3.12; running on", sys.version.split()[0])
!pip install -q -e . --no-deps {flag}
!pip install -q "numpy==2.5.3" "scipy==1.18.1" "pandas==3.0.6" "pyarrow==25.0.1"
!pip install -q "scikit-learn==1.9.1" "mne==1.13.2" "pymatreader==1.3.2" "pyriemann==0.12"
!pip install -q "braindecode==1.8.1" "onnx==1.23.1" "pydantic==2.13.5" "typer==0.27.2"
!pip install -q "pingouin==0.7.0" "statsmodels==0.15.0"
import torch

gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no gpu"
print(torch.__version__, gpu)

In [ ]:
!mkdir -p data/derived models/pretrained results/runs
!ln -sfn /kaggle/input/sixth-finger-bci-epochs data/derived/epochs
!cp /kaggle/input/sixth-finger-bci-pretrained/*.pt models/pretrained/ 2>/dev/null || true
# Resume: earlier outputs attached as inputs live under /kaggle/input/<slug>/out/ (read-only).
!cp -r /kaggle/input/*/out/runs/* results/runs/ 2>/dev/null || true
!cp -n /kaggle/input/*/out/pretrained/*.pt models/pretrained/ 2>/dev/null || true
import glob

import pandas as pd

resumed = sum(len(pd.read_parquet(f)) for f in glob.glob("results/runs/*/scores.parquet"))
print("resumed rows:", resumed, "(0 means a fresh start: is the previous output attached?)")

In [ ]:
!sixthfinger doctor
!sixthfinger run {EXPERIMENT} --methods {METHODS} --subjects {SUBJECTS}

In [ ]:
!mkdir -p /kaggle/working/out
!cp -r results/runs /kaggle/working/out/
!cp -r models/pretrained /kaggle/working/out/ 2>/dev/null || true
!ls -R /kaggle/working/out | head -50